# 5장 — 자의 끝: 오버플로·언더플로·NaN 🧗

웹 챕터의 세 실험(2배 여행 · exp 폭발 시계 · NaN 공장)을 코드로 재현하고,
자의 지도(정규·비정규·0·inf 구간)를 그립니다.

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter`를 누르며 위에서 아래로 내려가세요.

**이 노트북의 목표**: `loss: nan` 사건의 전말.
**"폭발 → 포화(inf) → 변이(NaN) → 전염"의 4단계를 직접 일으켜 본다.**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch5')

## 1. 2배 여행: 자의 양쪽 끝까지 몇 걸음?

1에서 출발해 fp32 로 ×2(그리고 ÷2)를 반복합니다.

In [ ]:
import numpy as np
f32 = np.float32

# 위로: ×2 여행 — inf 까지
x, n_up = f32(1.0), 0
with np.errstate(over='ignore'):                # 오버플로 경고는 오늘의 주인공이라 조용히
    while np.isfinite(x):
        x = f32(x * 2)
        n_up += 1
print(f'×2 여행: {n_up}번 만에 {x} (오버플로)')

# 아래로: ÷2 여행 — 0 까지 (비정규 경사로 포함)
y, n_down = f32(1.0), 0
tiny = float(np.finfo(np.float32).tiny)          # 최소 정규 ≈ 1.18e-38
passed_normal = None
while y > 0:
    y = f32(y / 2)
    n_down += 1
    if passed_normal is None and 0 < y < tiny:
        passed_normal = n_down                   # 비정규 구간 진입 시점
print(f'÷2 여행: {passed_normal}번째부터 비정규 경사로, {n_down}번 만에 0 (언더플로)')

assert n_up == 128, '위쪽 끝 — 2^128 이 최대(3.4×10³⁸)를 넘는 순간'
assert n_down == 150, '아래쪽 끝 — 마지막 눈금 2^−149(149번째)까지 버틴 뒤 150번째에 0'
assert abs(float(np.finfo(np.float32).max) - 3.4028235e38) < 1e31
print()
print('✔ 확인: 10³⁸ 도 ×2 로는 단 128걸음. 그리고 끝을 넘어도 에러 없이 inf 로 계속 달린다')

## 2. exp 폭발 시계: 로짓 89면 즉사

2부 7장의 지수함수는 자의 끝을 향해 전력질주합니다. fp32 와 fp64 의 폭발 지점을 실측.

In [ ]:
with np.errstate(over='ignore'):
    print(' x      fp32 exp(x)          fp64 exp(x)')
    for x in [10, 50, 88, 89, 709, 710]:
        e32 = np.exp(f32(x))
        e64 = np.exp(np.float64(x))
        print(f'{x:>4}   {str(e32):>16}   {str(e64):>16}')

    assert np.isfinite(np.exp(f32(88))), 'fp32: x=88 은 아직 유한(1.65×10³⁸)'
    assert np.isinf(np.exp(f32(89))), 'fp32: x=89 에서 inf — 어텐션 점수·로짓이 닿을 수 있는 크기!'
    assert np.isfinite(np.exp(np.float64(709))) and np.isinf(np.exp(np.float64(710))), 'fp64 경계는 709~710'
print()
print('✔ 확인: softmax 의 exp 를 그대로 계산하면 위험한 이유. 구급대(max 빼기)는 6장에서')

## 3. NaN 공장: 변이와 전염

자의 끝(inf·0)끼리 만나면 답이 없는 계산이 태어나고, 태어난 NaN 은 모든 것을 오염시킵니다.

In [ ]:
inf = np.float64(np.inf)
with np.errstate(invalid='ignore', divide='ignore'):
    cases = {
        '0 / 0': np.float64(0) / np.float64(0),
        'inf - inf': inf - inf,
        'sqrt(-1)': np.sqrt(np.float64(-1)),
        'inf / inf': inf / inf,
        '1 / 0 (주의!)': np.float64(1) / np.float64(0),
    }
for name, v in cases.items():
    print(f'{name:>14} = {v}')

assert all(np.isnan(v) for k, v in cases.items() if '주의' not in k), '답이 없는 계산 = NaN'
assert np.isinf(cases['1 / 0 (주의!)']), '1/0 은 NaN 이 아니라 inf — 헷갈리기 쉬운 예외'

# 전염성 — 하나만 끼어도 장부 전체가 오염
ledger = np.array([3.0, 5.0, np.nan])
print(f'\n3 + 5 + NaN = {ledger.sum()}   (전염!)   /   np.nansum = {np.nansum(ledger)} (NaN 무시 합)')
assert np.isnan(ledger.sum()) and np.nansum(ledger) == 8.0

# 자기부정 — NaN != NaN
nan = np.float64(np.nan)
assert not (nan == nan) and np.isnan(nan), 'NaN 은 자기 자신과도 다르다 — 검사는 isnan 으로'
print()
print('✔ 확인: 폭발 → 포화(inf) → 변이(NaN) → 전염, loss: nan 사건의 4단계')

## 4. 더 깊이 실측: 비정규 경사로와 자의 지도

지수 비트 전부 0 인 특별석: 숨은 1을 포기하고(0.m) 2⁻¹⁴⁹ 까지 버팁니다.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

tiny = float(np.finfo(np.float32).tiny)            # 최소 정규 2^-126 ≈ 1.18e-38
smallest = float(np.finfo(np.float32).smallest_subnormal)  # 2^-149 ≈ 1.4e-45
print(f'최소 정규     = {tiny:.6g} (= 2^-126)')
print(f'최소 비정규   = {smallest:.6g} (= 2^-149), 경사로의 마지막 눈금')
assert f32(2.0**-149) > 0 and f32(2.0**-150) == 0.0, '2^-149 는 살고 2^-150 은 0 으로'

# 자의 지도 — 로그 축 위의 구간들
fig, ax = plt.subplots(figsize=(7, 1.8))
ax.set_xscale('log'); ax.set_xlim(1e-50, 1e42); ax.set_yticks([])
ax.axvspan(smallest, tiny, color='tab:orange', alpha=0.35, label='비정규 (경사로)')
ax.axvspan(tiny, 3.4e38, color='tab:green', alpha=0.25, label='정규 구간')
ax.axvline(3.4e38, color='tab:red', ls='--'); ax.text(4e38, 0.4, '→ inf', color='tab:red')
ax.axvline(smallest, color='tab:orange', ls='--'); ax.text(2e-50, 0.4, '0 ←', color='tab:orange')
ax.legend(loc='upper left', fontsize=8); ax.set_title('fp32 자의 지도: 양수 쪽')
plt.tight_layout(); plt.show()
print('✔ 확인: 낭떠러지 대신 경사로. 대가는 정밀도(숨은 1 포기)와 일부 하드웨어의 속도')

**직접 해보기**: ① fp16 의 자로 2배 여행을 해 보세요. `np.finfo(np.float16).max` 는 65,504,
inf 까지 단 **16걸음**입니다(7장 loss scaling 의 동기). ② `np.float32(0.01) ** 300` 대신
확률 300개의 log 를 **더해** 보세요. 언더플로 없이 멀쩡한 이유가 6장의 주제입니다.

## 5. 관찰 정리 + 🤖 AI 연결

- fp32 자의 끝: 위 **3.4×10³⁸**(×2 로 128걸음), 아래 정규 **1.2×10⁻³⁸** → 비정규 경사로 →
  **2⁻¹⁴⁹** 에서 0. 끝을 넘어도 에러 없음. **inf 포화** 후 계속 달린다.
- **exp 는 단골 방아쇠**: fp32 는 x ≈ 89, fp64 는 x ≈ 710 에서 폭발. softmax 그대로는 위험(6장).
- **NaN**: 0/0·∞−∞·√(−1) 의 표식. 전염성(하나면 전체 오염) + 자기부정(NaN ≠ NaN, 검사는 isnan).
  1/0 은 inf(예외). 확률 연쇄 곱은 언더플로(log 피신이 6장).
- **AI 연결**: loss: nan = 폭발 → 포화 → 변이 → 전염의 결과(8부 9장 스파이크의 병리).
  처방 = 사슬 끊기: 학습률↓·gradient clipping(gradient = 그래디언트, **기울기**. 폭발 전에
  자르기, 8부 9장)·loss scaling(7장, fp16 자는 최대 65,504).

**다음 챕터**: 6장, 안전한 확률 계산(log-sum-exp). exp 폭발과 확률 언더플로를
한 번에 피하는 업계 표준 구급법, "max 빼기"의 수학.